# Análisis exploratorio de los datos

In [74]:
## importación de librerías y funciones
import sys
import os
# Le dice python que busque liberrías ahí también
sys.path.append(os.path.join('..', 'src'))

import streamlit as st
import pandas as pd
import scipy as sp
import plotly as px
import funciones_personales as fp

In [75]:
df_cars = pd.read_csv('../vehicles_us.csv')
df_cars.info()
print(df_cars.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  object 
 3   condition     51525 non-null  object 
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  object 
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  object 
 8   type          51525 non-null  object 
 9   paint_color   42258 non-null  object 
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  object 
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), object(7)
memory usage: 5.1+ MB
Index(['price', 'model_year', 'model', 'condition', 'cylinders', 'fuel',
       'odometer', 'transmission', 'type', 'paint_color', 'is_4wd',
       'date_posted', 'days_listed'],
     

##

## Análisis de NaN en el df

In [76]:
for columna in df_cars.columns:
    print(f"Total de valores únicos para {columna} es {df_cars[columna].nunique()}")
    unique_values = df_cars[columna].dropna().unique()
    if pd.api.types.is_numeric_dtype(df_cars[columna]):
        print(f"El valor minimo para {columna} es {df_cars[columna].min()}")
        print(f"El valor maximo para {columna} es {df_cars[columna].max()}")
        sorted_unique_values = sorted(unique_values)
        print(f"los valores únicos en orden ascendente son: \n{sorted_unique_values}")
    else:
        print(f"los valores únicos son: \n{unique_values}")
    fp.mostrar_nan(df_cars, columna, mostrar=True)

Total de valores únicos para price es 3443
El valor minimo para price es 1
El valor maximo para price es 375000
los valores únicos en orden ascendente son: 
[np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(15), np.int64(20), np.int64(24), np.int64(25), np.int64(28), np.int64(32), np.int64(35), np.int64(36), np.int64(39), np.int64(65), np.int64(69), np.int64(80), np.int64(85), np.int64(105), np.int64(111), np.int64(147), np.int64(155), np.int64(169), np.int64(171), np.int64(176), np.int64(179), np.int64(180), np.int64(185), np.int64(187), np.int64(188), np.int64(190), np.int64(195), np.int64(196), np.int64(197), np.int64(198), np.int64(199), np.int64(200), np.int64(204), np.int64(206), np.int64(211), np.int64(215), np.int64(222), np.int64(228), np.int64(233), np.int64(237), np.int64(243), np.int64(244), np.int64(245), np.int64(246), np.int64(247), np.int64(250), np.int64(253), np.int64(255), np.int64(257), np.int64(267)

 ## Observaciones de tipos de datos y valores NaN

 Columna `'price'`: 
 
 * Tipo de dato: int64 sin valores NaN.
 * Revisar filas de los valores de precio = 1 ¿que puede significar?

 Columna `'model_year'`:

 * Tipo de dato: Float64 con 3619 valores NaN.
    * Valores NaN deben ser valor desconocido para el.
    * Para decidir qué hacer con los NaN, es necesario investigar cómo trabaja plotly y scipy con los NaN. 
 * Se propone cambiar tipo de dato a int64.
    
 Columna `'model'`:

 * Tipo de dato string, solo 100 modelos diferentes 0 valores NaN.
 * A partir de esta columna se podría generar manufacturers.
    * Todos los valores de model comienzan por el nombre del fabricante (hyundai, chevrolet, ...) podemos separar la columna en dos o dejar model como está y crear una nueva que cocntenga la primera palabra de cada string en la columna `'model'`. 

 Columna `'condition'`:

 * Tipo de dato string, solo 7 diferentes: ['good' 'like new' 'fair' 'excellent' 'salvage' 'new'] sin NaN.

 Columna `'cylinders'`:

 * Tipo de dato: Float64, 5260 valores NaN.
    * Los NaN pueden significar valor desconocido del número de cilindros.
    * Los cilindros faltantes podrían obtenerse con el año y el modelo?
        * Con el año y el modelo de un auto generalmente se puede determinar el número de cilindros, aunque no siempre es definitivo si el modelo ofrecía múltiples opciones de motor (ej. 4 y 6 cilindros).
    * Para decidir qué hacer con los NaN, es necesario investigar cómo trabaja plotly y scipy con los NaN.
 * Se recomienda cambiar a int64.

 Columna `'fuel'`:

 * Tipo de dato: string ['gas' 'diesel' 'other' 'hybrid' 'electric']. Sin NaN.

 Columna `'odometer'`:
 
 * Tipo de dato: Float64 desde 0.0 a 990,000 (¿millas o km?). 7892 valores NaN.
    * Todos los valores 0 deben coinsidir con `'condition' == 'new'`
    * Para decidir qué hacer con los NaN, es necesario investigar cómo trabaja plotly y scipy con los NaN.
 * Se sugiere cambiar a tipo int64.

 Columna `'transmission'`:

 * Tipo de dato: String, ['automatic' 'manual' 'other'], Sin NaN.

 Columna `'type'`:
 
 * Tipo de dato: String, ['SUV' 'pickup' 'sedan' 'truck' 'coupe' 'van' 'convertible' 'hatchback' 'wagon' 'mini-van' 'other' 'offroad' 'bus'], sin NaN.

 Columna `'paint_color'`:

 * Tipo dato: String, ['white' 'red' 'black' 'blue' 'grey' 'silver' 'custom' 'orange' 'yellow' 'brown' 'green' 'purple'], 9267 valores NaN.
    * Valores NaN probablemente a color desconocido.
 * Se recomienda sustituir NaN por 'unknown'.

 Columna `'is_4wd'`:

 * Tipo de dato: Float64, Solo valores 1.0 o NaN.
 * Se recomienda cambiar a True or False.

 Columna `'date_posted'`:
 * Tipo de dato: String, sin NaN.
 * Se recomienda cambiar a tipo datetime.

 Columna `'days_listed'`:

 * Tipo de dato int64, de 0 a 271, sin NaN.

### Revisión de casos `'price' == 1` 

In [77]:
# filtra filas donde price = 1
df_cars_price_1 = df_cars[df_cars['price'] == 1]
print(f"Total de filas con price = 1: {len(df_cars_price_1)}")
# ¿Qué características tienen estos autos?
print("\nDistribución de condiciones para precio = 1:")
print(df_cars_price_1['condition'].value_counts())

print("\nDistribución de tipos para precio = 1:")
print(df_cars_price_1['type'].value_counts())
print(df_cars_price_1.head(10))

Total de filas con price = 1: 798

Distribución de condiciones para precio = 1:
condition
excellent    741
like new      27
good          18
fair           9
new            3
Name: count, dtype: int64

Distribución de tipos para precio = 1:
type
truck          287
SUV            250
sedan          141
coupe           74
van             16
convertible     11
hatchback        9
pickup           7
mini-van         2
other            1
Name: count, dtype: int64
      price  model_year                model  condition  cylinders    fuel  \
405       1      2014.0     chevrolet camaro  excellent        6.0     gas   
3063      1      1998.0  chevrolet silverado       good        8.0     gas   
3808      1      2007.0      chevrolet tahoe       good        8.0     gas   
3902      1      1996.0           ford f-150       fair        NaN     gas   
4140      1      2004.0  chevrolet silverado  excellent        8.0  diesel   
5612      1      2006.0           gmc sierra  excellent        NaN    

**Observaciones clave:**

* Los coches tienen años variados (1996-2015).
* Kilometrajes realistas (desde 200 hasta 192,960).
* Marcas y modelos específicos.
* Condiciones principalmente "excellent" y "good".
* Fechas de publicación normales

**Posibilidades a considerar:**

* ¿Podrían ser subastas o ventas especiales? Algunos sitios usan $1 como precio inicial.
* ¿Marcadores de "precio a consultar"? Cuando el vendedor no quiere mostrar el precio real.
* ¿Datos faltantes codificados como $1? Similar a los odómetros en 0.

**Conclusión**
Los datos se considerarán como precios ocultos o subastas y se sustituiran por NaN:

**Justificación:**

* Los estadísticos descriptivos (media, mediana, desviación estándar) los ignorarán automáticamente.
* Las visualizaciones no se distorsionarán por estos valores extremos.
* Se mantiene la integridad del análisis sin eliminar completamente los registros.

### Chequeo de hipótesis 'odometer' == 0 <=> condition == 'new'.


In [78]:
# Filtramos las filas en las que 'odometer' es igual a 0
df_cars_cero_odom = df_cars[df_cars['odometer'] == 0]
df_cars_cero_odom['condition'].unique()
# Autos con condition = 'new'
df_cars_new = df_cars[df_cars['condition'] == 'new']
print(f"Valores de odometer para autos 'new':")
print(df_cars_new['odometer'].describe())
print(len(df_cars_cero_odom))

Valores de odometer para autos 'new':
count       125.000000
mean      43476.056000
std       67269.684251
min           5.000000
25%          21.000000
50%        8002.000000
75%       69000.000000
max      315000.000000
Name: odometer, dtype: float64
185


**Conclusión**
Ya que ningun auto con `'condition'' == new'` coincide con  0 en `'odometer'`, y que varios autos con `'condition'' == new'` tienen en odometer rangos de 5 a 315,000. Los valores de cero deben ser datos faltantes.
Se recomienda cambiarlos a NaN.

## Limpieza y transformación de datos


In [79]:
# En la columna 'price', sustituir valores = 1 con NaN y convertir a int64
df_cars['price'] = df_cars['price'].replace(1, pd.NA).astype('Int64')
# transformar 'model_year' a int64
df_cars['model_year'] = df_cars['model_year'].astype('Int64')
# Filtra la primer palabra del string de la columna 'model' y añade una nueva columna 'manufacturers'
df_cars['make'] = df_cars['model'].str.split().str[0]
# borra la primer palabra del string de la columna 'model' y sustituye el valor original de 'model' con el nuevo string
df_cars['model'] = df_cars['model'].str.split().str[1:].apply(lambda x: ' '.join(x))
# transforma la columna 'cylinders' a int64
df_cars['cylinders'] = df_cars['cylinders'].astype('Int64')
# transformar datos de odometer a int64 y luego sustituir 0's con NaN
df_cars['odometer'] = df_cars['odometer'].replace(0, pd.NA).astype('Int64')
# sustituye NaN en 'paint_color' con 'unknown'
df_cars['paint_color'] = df_cars['paint_color'].fillna('unknown')
# transforma los valores de 'is_4wd' == 1 en True y NaN en False
df_cars['is_4wd'] = df_cars['is_4wd'].fillna(0).astype(bool)
# transforma los valores de 'date_posted' a datetime
df_cars['date_posted'] = pd.to_datetime(df_cars['date_posted'], errors='coerce')


### Reordenar columnas y exportar datos limpios

In [84]:
# Definir el orden de las columnas: principales primero, luego las auxiliares
columnas_ordenadas = [
    # Columnas principales para la app
    'price', 'model_year', 'make', 'model', 'condition', 
    'cylinders', 'fuel', 'odometer', 'transmission', 
    'type', 'paint_color', 'is_4wd',
    # Columnas auxiliares (no se mostrarán en la app pero se conservan)
    'date_posted', 'days_listed'
]

# Reordenar las columnas
df_cars_clean = df_cars[columnas_ordenadas].copy()

# Exportar todos los datos limpios
df_cars_clean.to_csv('vehicles_us_clean.csv', index=False)

# Verificar el resultado
print("Columnas en el orden final:")
print(df_cars_clean.columns.tolist())
print(f"Forma del dataset: {df_cars_clean.shape}")

Columnas en el orden final:
['price', 'model_year', 'make', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'transmission', 'type', 'paint_color', 'is_4wd', 'date_posted', 'days_listed']
Forma del dataset: (51525, 14)


# Descripción del desarrollo de la app.

se usará plotly, pandas, scipy y streamlit

Dataviewer con slides verticales y horizontales que muestran el dataframe.
* Columnas a mostrar en ese orden: [´price', 'model_year', 'make', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'transmission', 'type', 'paint_color', is_4wd ]
* Debe contener una checkbox que incluya a los fabricantes con mas de 1000 anuncios REVISAR 

Gráfico de barras "Tipos de vehículo por fabricante"
* Mostrará 'make' (fabricante) en eje x.
* Las barras mostrarán la cantidad de tipos de vehículo del fabricante con código de color.
* Cada tipo de vehículo podra seleccionarse desde la leyenda y al seleccionarlo este dejará de mostrarse en la gráfica.

Histograma de frecuencias "Año vs condición"
* Eje x mostrará 'model_year' (año).
* Mostrará un histograma de cada 'condition' (condición)
* Tambien la condición podra ser removida de la gráfica al dar click en el código de color de la leyenda.

Comparación de distribución de precios entre fabricantes.
* Debe tener dos cajas para seleccionar 'make' (fabricante) que se compararán.
* Debe tener un checkbox que cuando esté activado se muestre la gráfica normalizada (activado por defecto).
* Debe mostrar gráfico con el 'price' (precio en miles) y el porcentaje en el eje y.
